In [20]:
import os
import fitz
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import re
from collections import Counter

In [21]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zviad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zviad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [22]:
# Define PDF directory
pdf_folder = r"C:\Zviad_Thinkpad\Zviad\work\Healthcare data\Kaggle\Resume Dataset\data\data\Accountant"

In [23]:
# Extract text from PDFs
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = " ".join(page.get_text() for page in doc)
    return text if text.strip() else "empty_document"

In [24]:
resume_stopwords = {
    "summary", "profile", "objective", "experience", "education", "skills","work experience", "professional experience",
    "education", "projects", "interests", "languages", "contact", "profile", "created",
    "certifications", "references", "managed", "developed", "led","strong", "excellent", "dynamic", "motivated", "results-oriented",
    "hardworking", "team player", "detail-oriented", "reliable", "proven",
    "effective", "fast learner","year", "years", "month", "months", "present", "current", "date",
    "from", "to", "until",
    "worked", "provided", "assisted", "collaborated", "cv", "resume",
    "applicant", "position", "role", "responsibilities", "employment",
    "year", "month", "date", "location", "address", "phone", "linkedin", "github", "website"
}

In [25]:
# Minimal preprocessing for high-frequency analysis
def minimal_preprocessing(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    tokens = word_tokenize(text)
    return tokens

In [26]:
# Collect PDFs and preprocess minimally
pdf_files = [os.path.join(pdf_folder, f) for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
corpus = [minimal_preprocessing(extract_text_from_pdf(pdf)) for pdf in pdf_files]

In [27]:
# Set hyperparameter ranges
num_topics_range = [4, 6, 8, 10]   # Adjust topic count
num_passes_range = [10, 15, 20]    # Adjust training passes
num_most_freq_words_range = [50, 100, 150]  # Adjust most frequent words for stopwords

In [28]:
# Run grid search over hyperparameters
best_score = 0
best_params = None

In [29]:
for num_topics in num_topics_range:
    for num_passes in num_passes_range:
        for num_most_freq_words in num_most_freq_words_range:
            
            # Extract high-frequency words dynamically
            word_counts = Counter(word for doc in corpus for word in doc)
            high_freq_terms = {word for word, count in word_counts.most_common(num_most_freq_words)}

            # Merge industry-specific and standard stopwords
            standard_stopwords = set(stopwords.words("english"))
            filtered_stopwords = standard_stopwords.union(high_freq_terms).union(resume_stopwords)

            # Final preprocessing with stopwords
            def final_preprocessing(tokens):
                return [word for word in tokens if word.isalnum() and word not in filtered_stopwords]

            corpus_filtered = [final_preprocessing(doc) for doc in corpus]

            # Convert text into LDA format
            dictionary = corpora.Dictionary(corpus_filtered)
            doc_term_matrix = [dictionary.doc2bow(doc) for doc in corpus_filtered]

            # Train LDA model with current hyperparameters
            lda_model = gensim.models.LdaModel(doc_term_matrix, num_topics=num_topics, id2word=dictionary, passes=num_passes, alpha='auto', eta='auto')

            # Compute coherence score
            coherence_model = CoherenceModel(model=lda_model, texts=corpus_filtered, dictionary=dictionary, coherence='c_v')
            coherence_score = coherence_model.get_coherence()

            # Print results for each combination
            print(f"Topics: {num_topics}, Passes: {num_passes}, Most Frequent Words: {num_most_freq_words} → Coherence Score: {coherence_score:.4f}")

            # Track best parameters
            if coherence_score > best_score:
                best_score = coherence_score
                best_params = (num_topics, num_passes, num_most_freq_words)

Topics: 4, Passes: 10, Most Frequent Words: 50 → Coherence Score: 0.2975
Topics: 4, Passes: 10, Most Frequent Words: 100 → Coherence Score: 0.2928
Topics: 4, Passes: 10, Most Frequent Words: 150 → Coherence Score: 0.2939
Topics: 4, Passes: 15, Most Frequent Words: 50 → Coherence Score: 0.2998
Topics: 4, Passes: 15, Most Frequent Words: 100 → Coherence Score: 0.3026
Topics: 4, Passes: 15, Most Frequent Words: 150 → Coherence Score: 0.2783
Topics: 4, Passes: 20, Most Frequent Words: 50 → Coherence Score: 0.3061
Topics: 4, Passes: 20, Most Frequent Words: 100 → Coherence Score: 0.2605
Topics: 4, Passes: 20, Most Frequent Words: 150 → Coherence Score: 0.2818
Topics: 6, Passes: 10, Most Frequent Words: 50 → Coherence Score: 0.2945
Topics: 6, Passes: 10, Most Frequent Words: 100 → Coherence Score: 0.3220
Topics: 6, Passes: 10, Most Frequent Words: 150 → Coherence Score: 0.2806
Topics: 6, Passes: 15, Most Frequent Words: 50 → Coherence Score: 0.3161
Topics: 6, Passes: 15, Most Frequent Words:

In [30]:
# Print best hyperparameters
print(f"\nBest LDA Configuration → Topics: {best_params[0]}, Passes: {best_params[1]}, Most Frequent Words: {best_params[2]} → Coherence Score: {best_score:.4f}")


Best LDA Configuration → Topics: 10, Passes: 10, Most Frequent Words: 150 → Coherence Score: 0.3267
